# Entity ***Swaps***
+ Layer **bronze**
+ Priority **2**
---
### Imports

In [0]:
%run ../common/bronze_constants

In [0]:
%run ../common/medallion_functions

In [0]:
import requests
import time
from datetime import datetime, timedelta, timezone
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Subgraph API

In [0]:
SUBGRAPH_URL = get_subgraph_api_url_from_secrets()

### Variables

In [0]:
days = 7
swaps_list = []
query_variables= {
    "batch_pool_ids": [],
    "skip": 0,
    "since_ts": int((datetime.now(timezone.utc)- timedelta(days=days)).timestamp())
}

In [0]:
swaps_query = """
query SwapsQuery($batch_pool_ids: [String!]!, $skip: Int!, $since_ts: BigInt!){
  swaps(
    where: {
      pool_in: $batch_pool_ids,
      amountUSD_not_in: ["0"],
      timestamp_gt: $since_ts
    }
    first: 1000,
    skip: $skip
  ) {
    id
    transaction {
      id
    }
    pool {
      id
    }
    token0{
      id
    }
    token1{
      id
    }
    amount0
    amount1
    amountUSD
    sender
    recipient
    tick
    timestamp
  }
}
"""

### Filtering relevant pools and time period
+ Swaps could overload the graph queries threshold since within a single pool could exists thousands of swaps.

In [0]:
filtered_bz_pools_df = retrieve_relevant_bronze_pools(
                            tvl_usd = 50000,
                            vol_usd = 10000,
                            tx_count = 100
                            )

### Execution

In [0]:
pool_ids_list = [row.id for row in filtered_bz_pools_df.select("id").collect()]

In [0]:
for i in range(0, len(pool_ids_list), BATCH_SIZE):
    batch_pools = pool_ids_list[i:i+BATCH_SIZE]
    query_variables["batch_pool_ids"] = batch_pools
    query_variables["skip"] = 0
    print(f"*INFO*: Loading swaps for {len(batch_pools)} pools.")
    while True:
        response = requests.post(SUBGRAPH_URL, json={"query": swaps_query, "variables": query_variables})
        swaps_data = response.json()["data"][entity_name]

        if "errors" in response.json():
            raise Exception(response.json()["errors"])

        swaps_list.extend(swaps_data)

        time.sleep(0.5)

        rows_loaded = len(swaps_data)
        print(f"*INFO*: Loaded rows: {rows_loaded}.")

        if rows_loaded < PAGE_SIZE:
            break

        query_variables["skip"] += PAGE_SIZE

In [0]:
swaps_df = spark.createDataFrame(swaps_list)
swaps_df = swaps_df.withColumn("_load_timestamp_bronze", current_timestamp())

### Save & exit

In [0]:
load_result = load_entity(swaps_df,
                        entity_name,
                        load_pattern,
                        layer
                        #,
                        )

In [0]:
dbutils.notebook.exit(load_result)